In [1]:
# ── SETUP ──────────────────────────────────────────────────────────────────────
import sqlite3
import pandas as pd

# Connect to database
conn = sqlite3.connect('eu_crime.db')
print("Connected to eu_crime.db")

Connected to eu_crime.db


In [2]:
# ── DATASET OVERVIEW ───────────────────────────────────────────────────────────

query = """
SELECT 'Total records' AS metric, COUNT(*) AS value FROM crimes
UNION ALL
SELECT 'Countries', COUNT(DISTINCT country) FROM crimes
UNION ALL
SELECT 'Crime types', COUNT(DISTINCT crime_type) FROM crimes
UNION ALL
SELECT 'Year range start', MIN(year) FROM crimes
UNION ALL
SELECT 'Year range end', MAX(year) FROM crimes
"""

pd.read_sql(query, conn)

,metric,value
0,Total records,2452
1,Countries,36
2,Crime types,7
3,Year range start,2008
4,Year range end,2021


In [3]:
# ── TOP 10 COUNTRIES BY TOTAL CRIMES ──────────────────────────────────────────

query = """
SELECT country, 
       SUM(count) AS total_crimes
FROM crimes
GROUP BY country
ORDER BY total_crimes DESC
LIMIT 10
"""

pd.read_sql(query, conn)

,country,total_crimes
0,Italy,18014525.0
1,Germany,15706996.0
2,Sweden,9134034.0
3,Türkiye,8492088.0
4,Spain,8067528.0
5,Netherlands,6563194.0
6,Denmark,5106245.0
7,Belgium,4315408.0
8,Austria,3847273.0
9,Switzerland,3443563.0


In [4]:
# ── HOMICIDE RISK LEVELS ───────────────────────────────────────────────────────

query = """
SELECT country,
       ROUND(AVG(count), 1) AS avg_homicides,
       CASE 
           WHEN AVG(count) < 50  THEN 'Low'
           WHEN AVG(count) < 200 THEN 'Medium'
           ELSE 'High'
       END AS risk_level
FROM crimes
WHERE crime_type = 'Intentional Homicide'
GROUP BY country
ORDER BY avg_homicides DESC
"""

pd.read_sql(query, conn)

,country,avg_homicides,risk_level
0,Türkiye,2024.4,High
1,Germany,655.6,High
2,Italy,446.9,High
3,Spain,336.1,High
4,Romania,253.1,High
5,Belgium,189.2,Medium
6,Lithuania,138.8,Medium
7,Greece,115.3,Medium
8,Latvia,110.3,Medium
9,Bulgaria,109.1,Medium


In [5]:
# ── EU vs NON-EU: AVERAGE HOMICIDE RATE ───────────────────────────────────────

query = """
SELECT co.eu_member,
       ROUND(AVG(c.count), 1) AS avg_homicides
FROM crimes c
INNER JOIN countries co ON c.country_code = co.country_code
WHERE c.crime_type = 'Intentional Homicide'
GROUP BY co.eu_member
"""

pd.read_sql(query, conn)

,eu_member,avg_homicides
0,No,274.0
1,Yes,134.0


In [6]:
# ── COVID IMPACT ON THEFT ──────────────────────────────────────────────────────

query = """
SELECT co.region,
       ROUND(AVG(CASE WHEN c.year < 2020 THEN c.count END), 0) AS avg_pre_covid,
       ROUND(AVG(CASE WHEN c.year >= 2020 THEN c.count END), 0) AS avg_post_covid
FROM crimes c
INNER JOIN countries co ON c.country_code = co.country_code
WHERE c.crime_type = 'Theft'
GROUP BY co.region
ORDER BY avg_pre_covid DESC
"""

pd.read_sql(query, conn)

,region,avg_pre_covid,avg_post_covid
0,Western Europe,310364.0,112724.0
1,Northern Europe,176476.0,118996.0
2,Southern Europe,156364.0,100669.0
3,Eastern Europe,49641.0,33135.0


In [7]:
# ── COUNTRIES ABOVE EUROPEAN AVERAGE FOR ROBBERY ──────────────────────────────

query = """
SELECT country, ROUND(AVG(count), 0) AS avg_robbery
FROM crimes
WHERE crime_type = 'Robbery'
GROUP BY country
HAVING AVG(count) > (
    SELECT AVG(count)
    FROM crimes
    WHERE crime_type = 'Robbery'
)
ORDER BY avg_robbery DESC
"""

pd.read_sql(query, conn)

,country,avg_robbery
0,Spain,69715.0
1,Germany,42594.0
2,Italy,29070.0
3,Belgium,19100.0
4,Portugal,14210.0
5,Netherlands,10544.0
6,Türkiye,10050.0
7,Sweden,8773.0
8,Poland,7637.0


In [8]:
# ── MOST COMMON CRIME TYPE PER COUNTRY ────────────────────────────────────────

query = """
SELECT country, crime_type, total
FROM (
    SELECT country, crime_type,
           SUM(count) AS total,
           RANK() OVER (PARTITION BY country ORDER BY SUM(count) DESC) AS rnk
    FROM crimes
    GROUP BY country, crime_type
)
WHERE rnk = 1
ORDER BY total DESC
"""

pd.read_sql(query, conn)

,country,crime_type,total
0,Italy,Theft,11663799.0
1,Germany,Theft,9805645.0
2,Sweden,Theft,5670026.0
3,Netherlands,Theft,3677120.0
4,Denmark,Theft,3267667.0
5,Türkiye,Theft,2973461.0
6,Switzerland,Theft,2063128.0
7,Austria,Theft,1982236.0
8,Spain,Theft,1887831.0
9,Finland,Theft,1874050.0


In [9]:
# ── CLOSE CONNECTION ───────────────────────────────────────────────────────────
conn.close()
print("Done!")

Done!
